# Generate Behavior Dataset With Category Logic

Notebook này sinh dữ liệu hành vi sao cho `product_id`, `action`, `category` và `session intent` có quan hệ logic hơn.
Mục tiêu là để model học được tín hiệu thật hơn thay vì cột `category` bị gán ngẫu nhiên.


In [ ]:
import random
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)


In [ ]:
CATEGORIES = ["Audio", "Laptop", "Monitor", "Accessories", "Storage"]
PRODUCTS_PER_CATEGORY = 20
NUM_PRODUCTS = len(CATEGORIES) * PRODUCTS_PER_CATEGORY

PRODUCT_CATEGORY_MAP = {}
for idx, category in enumerate(CATEGORIES):
    start_id = idx * PRODUCTS_PER_CATEGORY + 1
    end_id = start_id + PRODUCTS_PER_CATEGORY
    for product_id in range(start_id, end_id):
        PRODUCT_CATEGORY_MAP[product_id] = category

product_df = pd.DataFrame({
    "product_id": list(PRODUCT_CATEGORY_MAP.keys()),
    "category": list(PRODUCT_CATEGORY_MAP.values()),
})

product_df.head()


In [ ]:
SESSION_TEMPLATES = {
    "Audio": [
        ["view", "click", "add_to_cart", "purchase"],
        ["search", "view", "click", "wishlist"],
        ["view", "click", "purchase", "review"],
    ],
    "Laptop": [
        ["search", "view", "click", "add_to_cart"],
        ["view", "click", "wishlist", "purchase"],
        ["search", "view", "click", "remove_from_cart"],
    ],
    "Monitor": [
        ["view", "view", "click", "purchase"],
        ["search", "view", "click", "wishlist"],
        ["view", "click", "add_to_cart", "remove_from_cart"],
    ],
    "Accessories": [
        ["view", "click", "add_to_cart", "purchase", "review"],
        ["view", "wishlist", "purchase"],
        ["search", "view", "click", "add_to_cart"],
    ],
    "Storage": [
        ["search", "view", "click", "purchase"],
        ["view", "click", "add_to_cart", "purchase"],
        ["search", "view", "click", "remove_from_cart"],
    ],
}

ACTION_CATEGORY_WEIGHTS = {
    "view": {"primary": 0.72, "secondary": 0.20, "cross": 0.08},
    "search": {"primary": 0.60, "secondary": 0.30, "cross": 0.10},
    "click": {"primary": 0.80, "secondary": 0.15, "cross": 0.05},
    "add_to_cart": {"primary": 0.90, "secondary": 0.08, "cross": 0.02},
    "purchase": {"primary": 0.92, "secondary": 0.06, "cross": 0.02},
    "wishlist": {"primary": 0.85, "secondary": 0.10, "cross": 0.05},
    "remove_from_cart": {"primary": 0.88, "secondary": 0.08, "cross": 0.04},
    "review": {"primary": 0.95, "secondary": 0.04, "cross": 0.01},
}


In [ ]:
def category_products(category):
    return product_df.loc[product_df["category"] == category, "product_id"].tolist()


def choose_secondary_category(primary_category):
    candidates = [c for c in CATEGORIES if c != primary_category]
    return random.choice(candidates)


def choose_product(primary_category, secondary_category, action, last_product_id=None):
    weights = ACTION_CATEGORY_WEIGHTS[action]
    bucket = random.choices(
        population=["primary", "secondary", "cross"],
        weights=[weights["primary"], weights["secondary"], weights["cross"]],
        k=1,
    )[0]

    if action in {"add_to_cart", "purchase", "remove_from_cart", "review"} and last_product_id is not None:
        return last_product_id

    if bucket == "primary":
        chosen_category = primary_category
    elif bucket == "secondary":
        chosen_category = secondary_category
    else:
        chosen_category = random.choice(CATEGORIES)

    return random.choice(category_products(chosen_category))


In [ ]:
def generate_user_sequence(user_id, num_sessions=None):
    sequence = []
    session_count = num_sessions or random.randint(3, 10)
    cursor_time = datetime.now()

    for session_id in range(session_count):
        primary_category = random.choice(CATEGORIES)
        secondary_category = choose_secondary_category(primary_category)
        pattern = random.choice(SESSION_TEMPLATES[primary_category])
        last_product_id = None

        for step, action in enumerate(pattern):
            product_id = choose_product(
                primary_category=primary_category,
                secondary_category=secondary_category,
                action=action,
                last_product_id=last_product_id,
            )
            last_product_id = product_id
            sequence.append({
                "user_id": user_id,
                "product_id": product_id,
                "action": action,
                "timestamp": cursor_time + timedelta(minutes=session_id, seconds=step),
                "category": PRODUCT_CATEGORY_MAP[product_id],
                "session_primary_category": primary_category,
            })

        cursor_time += timedelta(minutes=5)

    return sequence


In [ ]:
data = []
NUM_USERS = 500

for user_id in range(1, NUM_USERS + 1):
    data.extend(generate_user_sequence(user_id))

df = pd.DataFrame(data)
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)
df.head(20)


In [ ]:
print(df["action"].value_counts(normalize=True).round(4))
print()
print(df["category"].value_counts(normalize=True).round(4))
print()
print(pd.crosstab(df["action"], df["category"], normalize="index").round(3))


In [ ]:
export_df = df[["user_id", "product_id", "action", "timestamp", "category"]].copy()
export_df.to_csv("data_user500.csv", index=False)
export_df.head()
